# Does AlexNet show human-like perceptual effects?

**Reading:** Jacob, G., Pramod, R. T., Katti, H., & Arun, S. P. (2021). Qualitative similarities and differences in visual object representations between brains and deep networks. *Nature Communications, 12*, 1872.

Jacob et al. tested whether deep networks show 13 classic effects from human perception and neuroscience, in a **trained** and an **untrained** network. Today you test two of them in AlexNet:

1. **Scene incongruence**
2. **Correlated sparseness**

**Teamwork:** your partner has a different notebook. Section 1 is the same for both of you, section 2 is different.

**How to use this notebook:** settings in CAPITALS (e.g. `OBJECT_SIZE`) can be changed. Change them and run the cell again.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import csv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch
from torchvision import models, transforms
device = "cpu"

**If Sciebo doesn't work:** download the Week 5 folder `scene_incongruence` and `animals_objects` from Sciebo, and upload it into a folder called `images` inside this notebook's folder (`Cognitive_Neuroscience_Update/Week5-Jacob2021`).

In [ ]:
# Find the course data: on JupyterHub it is in Sciebo, on your own computer next to this notebook
def find(sciebo_folder, local_folder):
    hits = sorted(Path.home().glob(f"sciebo/*/{sciebo_folder}"))
    folder = hits[0] if hits else Path(local_folder)
    if not folder.is_dir() or not any(folder.iterdir()):
        raise FileNotFoundError(
            f"Images were not found in {folder}.\n"
            "On JupyterHub: did you connect Sciebo when starting the server, and accept the shared CNU folder?\n"
            f"On your own computer: put the folder at {local_folder}"
        )
    return folder

SCENE_DIR = find("Week5/images/scene_incongruence", "images/scene_incongruence")
IMAGE_DIR = find("Week5/images/animals_objects", "images/animals_objects")         # 48 of last week's images

In [ ]:
# The trained and the untrained AlexNet, as last week
trained = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).to(device).eval()
torch.manual_seed(0)    # the same "random" weights for everyone
untrained = models.alexnet(weights=None).to(device).eval()
NETWORKS = {"trained": trained, "untrained": untrained}
CATEGORIES = models.AlexNet_Weights.IMAGENET1K_V1.meta["categories"]    # the 1,000 ImageNet names

# All images are prepared as 224 x 224 pixels; this turns them into network input
normalise = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

LAYERS = {
    "conv1": ("features", 1), "conv2": ("features", 4), "conv3": ("features", 7),
    "conv4": ("features", 9), "conv5": ("features", 11),
    "fc6": ("classifier", 2), "fc7": ("classifier", 5), "fc8": ("classifier", 6),
}

def get_acts(model, imgs):
    # Return a dict: layer name -> activations, one row per image (flattened)
    saved = {}
    hooks = [getattr(model, part)[i].register_forward_hook(
                 lambda layer, inp, out, name=name: saved.__setitem__(name, out.flatten(1).numpy()))
             for name, (part, i) in LAYERS.items()]
    with torch.no_grad():
        model(torch.stack([normalise(img.convert("RGB")) for img in imgs]).to(device))
    for h in hooks:
        h.remove()
    return saved

def plot_layers(values, title, ylabel):
    # One line per network across the 8 layers
    plt.figure(figsize=(7, 3.5))
    for net, v in values.items():
        plt.plot(list(LAYERS), v, marker="o", label=net)
    plt.axhline(0, color="grey", lw=0.8)
    plt.ylabel(ylabel); plt.title(title); plt.legend(); plt.show()

## 1. Scene incongruence

People recognise an object faster and more accurately in a scene where it belongs (a soccer ball in a stadium) than in one where it doesn't (a soccer ball in a factory). Does the network?

We paste 49 objects onto scenes that people rated as fitting (**congruent**) or not fitting (**incongruent**), and check whether the **trained** network still names the object. The untrained network can't name anything, so it is left out here.

*Images: ObScene database (Andrade et al., 2023).*

In [ ]:
pairs = list(csv.DictReader(open(SCENE_DIR / "pairs.csv")))     # object, correct labels, condition, scene
correct = {p["object"][:-4]: p["imagenet_labels"].split("; ") for p in pairs}
print(f"{len(correct)} objects, {len(pairs)} object-scene pairs")
print(sorted(correct))

def paste(obj, scene_name, size):
    # Paste the object in the centre; size = share of the image width/height
    cut = Image.open(SCENE_DIR / "objects" / f"{obj}.png")
    s = size * 224 / max(cut.size)
    cut = cut.resize((max(1, int(cut.width * s)), max(1, int(cut.height * s))))
    if scene_name == "white":
        bg = Image.new("RGB", (224, 224), "white")
    else:
        bg = what_network_sees(Image.open(SCENE_DIR / "scenes" / scene_name).convert("RGB")).resize((224, 224))
    bg.paste(cut, ((224 - cut.width) // 2, (224 - cut.height) // 2), cut)
    return bg

def top5(imgs):
    # The trained network's 5 best guesses for each image
    with torch.no_grad():
        out = trained(torch.stack([normalise(i) for i in imgs]).to(device))
    return [[CATEGORIES[k] for k in row.topk(5).indices] for row in out]

Change the examples to see more images & change the object size to see whether this has an effect on the results

In [ ]:
EXAMPLES = ["soccer_ball", "clock", "laundry_basket"]   # pick from the list above
OBJECT_SIZE = 0.7                                    # 0.7 = the object fills 70% of the image

fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(9, 3.2 * len(EXAMPLES)), squeeze=False)
for r, obj in enumerate(EXAMPLES):
    scenes = ["white"] + [next(p["scene"] for p in pairs if p["object"] == f"{obj}.png" and p["condition"] == c)
                          for c in ["congruent", "incongruent"]]
    imgs = [paste(obj, s, OBJECT_SIZE) for s in scenes]
    for c, (img, guess, s) in enumerate(zip(imgs, top5(imgs), scenes)):
        ok = guess[0] in correct[obj]
        axes[r, c].imshow(img); axes[r, c].axis("off")
        label = "white" if s == "white" else f"{['white', 'congruent', 'incongruent'][c]}: {s.split('.')[0]}"
        axes[r, c].set_title(f"{label}\nguess: {guess[0]}",
                             fontsize=8, color="green" if ok else "red")
plt.tight_layout(); plt.show()

Now all 49 objects, in congruent vs incongruent scenes. **Top-1:** the network's best guess is correct. **Top-5:** the correct name is among its 5 best guesses.

In [ ]:
OBJECT_SIZE = 0.7

def accuracy(obj, scene_names):
    guesses = top5([paste(obj, s, OBJECT_SIZE) for s in scene_names])
    return (np.mean([g[0] in correct[obj] for g in guesses]),
            np.mean([any(x in correct[obj] for x in g) for g in guesses]))

results = {"congruent": [], "incongruent": []}
for obj in correct:
    for cond in ["congruent", "incongruent"]:
        results[cond].append(accuracy(obj, [p["scene"] for p in pairs if p["object"] == f"{obj}.png" and p["condition"] == cond]))
acc = {cond: np.mean(r, axis=0) * 100 for cond, r in results.items()}    # average over objects, in %

for cond, (t1, t5) in acc.items():
    print(f"{cond:12s} top-1 {t1:3.0f}%   top-5 {t5:3.0f}%")
x = np.arange(2)
plt.figure(figsize=(5, 3.5))
plt.bar(x - 0.2, [a[0] for a in acc.values()], 0.4, label="top-1")
plt.bar(x + 0.2, [a[1] for a in acc.values()], 0.4, label="top-5")
plt.xticks(x, list(acc)); plt.ylabel("correct (%)"); plt.ylim(0, 100); plt.legend(); plt.show()

## 2. Correlated sparseness

Some neurons in monkey IT respond to only a few images (they are **sparse**, or selective), others respond to many. A neuron that is selective for shapes tends to be selective for textures too: selectivity is a property of the neuron, not of the images.

Jacob et al. tested this in each layer: they measured every unit's **sparseness** for one image set and for another, then **correlated** the two across units.

- **Sparseness** of a unit: 0 = responds equally to all images, 1 = responds to only one image.
- **Correlation r** across units: high if units that are selective for one image set are also selective for the other.

We use two sets of 64 images: **textures** (zoomed-in patches of last week's images) and **shapes** (black letters and symbols).

In [ ]:
rng = np.random.default_rng(0)
paths = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"} and not p.name.startswith("."))
textures = []
for k in range(64):                                   # 64 patches from 48 images: some images give two
    p = paths[k % len(paths)]
    x, y = rng.integers(40, 120, 2)
    img = what_network_sees(Image.open(p)).convert("L")
    textures.append(img.crop((x, y, x + 64, y + 64)).resize((224, 224)).convert("RGB"))

FONT = Path(matplotlib.get_data_path()) / "fonts/ttf/DejaVuSans-Bold.ttf"
font = ImageFont.truetype(str(FONT), 150)
shapes = []
for ch in "ABCDEFGHJKLMNPQRSTUVWXYZ2345679&%#@?$abdefghkmnqrtyWMXZ"[:64]:
    img = Image.new("RGB", (224, 224), "white"); d = ImageDraw.Draw(img)
    l, t, r, b = d.textbbox((0, 0), ch, font=font)
    d.text((112 - (l + r) / 2, 112 - (t + b) / 2), ch, font=font, fill="black")
    shapes.append(img)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for k in range(8):
    axes[0, k].imshow(textures[k]); axes[1, k].imshow(shapes[k])
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_title("textures", loc="left"); axes[1, 0].set_title("shapes", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
def sparseness(R):
    # R: images x units. Scale each unit to 0..1, then measure how few images it responds to
    R = (R - R.min(0)) / (R.max(0) - R.min(0) + 1e-12)
    n = R.shape[0]
    return (1 - R.mean(0) ** 2 / ((R ** 2).mean(0) + 1e-12)) / (1 - 1 / n)

spars, corr = {}, {}
for net, model in NETWORKS.items():
    a_tex, a_shp = get_acts(model, textures), get_acts(model, shapes)
    spars[net], corr[net] = {}, []
    for layer in LAYERS:
        # only units that respond differently to at least some images in both sets
        use = (a_tex[layer].std(0) > 1e-3 * a_tex[layer].std()) & (a_shp[layer].std(0) > 1e-3 * a_shp[layer].std())
        s_tex, s_shp = sparseness(a_tex[layer][:, use]), sparseness(a_shp[layer][:, use])
        spars[net][layer] = (s_tex, s_shp)
        corr[net].append(np.corrcoef(s_tex, s_shp)[0, 1])

plot_layers(corr, "Correlated sparseness", "correlation r (textures vs shapes)")

Each dot is one unit: its sparseness for textures against its sparseness for shapes.

In [ ]:
LAYER = "conv5"

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for ax, net in zip(axes, NETWORKS):
    s_tex, s_shp = spars[net][LAYER]
    ax.scatter(s_tex, s_shp, s=3, alpha=0.3)
    ax.set_title(f"{net}: {LAYER}  (r = {np.corrcoef(s_tex, s_shp)[0, 1]:.2f})")
    ax.set_xlabel("sparseness for textures"); ax.set_ylabel("sparseness for shapes")
plt.tight_layout(); plt.show()

#### Q1. Scene incongruence was measured from the network's guesses, correlated sparseness from the activations inside the network. What can each kind of measure tell you that the other can't?

## 3. Bringing it together (with your partner)
#### Q2. Put your two plots from section 2 next to each other. How does training change each effect, and in which layers? Do these effects come from the network's architecture, from its training, or both?